# Discriminative Imputation Experiments

**Two experiments testing the discriminative imputation hypothesis:**

**Experiment 1: Discriminative Exploration (DX)** — SAFE
- Extracts K closest alternative paths via single-edge removal
- Boosts UCB scores on discriminating edges (symmetric difference w/ alternatives)
- Gradient strictly masked on unobserved edges
- Tests: does targeted exploration of decision-boundary edges beat CX?

**Experiment 2: Discriminative Imputation (DI)** — RISKY
- Standard UCB action selection
- Adversarial imputation on discriminating edges per alternative:
  - Pessimistic (μ + ασ) on optimal-only edges
  - Optimistic (μ − ασ) on alternative-only edges
  - Gradient averaged over K alternatives, allowed on observed + discriminating edges
- Tests: does structured imputation avoid the gradient-through-imputed failure?

**Baselines:** BayesianMSE, UCB_SPO_TSMask, UCB_SPO_ModelMask, TS_SPO_TSMask, UCB_CX_a10

In [45]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import jax
import jax.numpy as jnp
import optax
import pandas as pd
from lets_plot import *

LetsPlot.setup_html()

from src.envs.synthetic import SyntheticKnapsackEnv
from src.envs.shortest_path import ShortestPathEdgeEnv
from src.contested import contested_mask_and_slacks
from src.agents.base import run_experiment
from src.solver import solve_knapsack, compute_edge_slacks, solve_shortest_path_edges
from src.agents.bayesian_utils import (
    init_per_arm_regression,
    compute_ucb_values,
    compute_bayesian_predictions,
    update_bayesian_stats,
)

## Loss Gradient Functions

In [46]:
def spo_grad(c_pred, c_true, solver, is_maximize=True):
    """SPO+ gradient w.r.t. predictions."""
    cp = np.asarray(c_pred, dtype=np.float64)
    ct = np.asarray(c_true, dtype=np.float64)
    n = cp.shape[0]

    w_true = solver(ct)
    w_spo = solver(2 * cp - ct)

    if is_maximize:
        return jnp.array(2.0 * (w_spo - w_true) / n, dtype=jnp.float32)
    else:
        return jnp.array(2.0 * (w_true - w_spo) / n, dtype=jnp.float32)


def mse_grad(c_pred, c_true):
    """MSE gradient w.r.t. predictions."""
    n = c_pred.shape[0]
    return 2.0 * (c_pred - c_true) / n


def make_spo_grad(solver, is_maximize=True):
    return lambda c_pred, c_true: spo_grad(c_pred, c_true, solver, is_maximize)


def make_inv_sqrt_schedule(init_lr):
    """LR schedule: lr_0 / sqrt(t+1). Same for all agents — no per-method tuning."""
    def schedule(count):
        return init_lr / jnp.sqrt(count + 1)
    return schedule

## JIT Kernels

In [47]:
@jax.jit
def _fused_ucb(X, W1, b1, A_inv, b_bayes, beta):
    h = X @ W1 + b1
    theta = jnp.einsum('nij,nj->ni', A_inv, b_bayes)
    means = theta @ h
    Ax = jnp.einsum('nij,j->ni', A_inv, h)
    stds = jnp.sqrt(jnp.maximum(jnp.einsum('i,ni->n', h, Ax), 0.0))
    return means + beta * stds

@jax.jit
def _fused_predict(X, W1, b1, A_inv, b_bayes):
    h = X @ W1 + b1
    theta = jnp.einsum('nij,nj->ni', A_inv, b_bayes)
    return theta @ h

@jax.jit
def _fused_bayesian_core(X, W1, b1, A, b_bayes, A_inv, rewards, mask):
    h = X @ W1 + b1
    A_new, b_new, A_inv_new = update_bayesian_stats(
        A, b_bayes, A_inv, h, rewards, mask)
    theta = jnp.einsum('nij,nj->ni', A_inv_new, b_new)
    pred = theta @ h
    return h, pred, theta, A_new, b_new, A_inv_new

@jax.jit
def _bayesian_grads(X, theta, grad_pred):
    grad_h = jnp.einsum('ni,n->i', theta, grad_pred)
    return {'W1': jnp.outer(X, grad_h), 'b1': grad_h}

def _make_step_fn(optimizer):
    @jax.jit
    def step(grads, opt_state, params):
        updates, new_state = optimizer.update(grads, opt_state, params)
        return optax.apply_updates(params, updates), new_state
    return step

In [48]:
def extract_k_alternatives(pred_costs, grid_size, K=5):
    """Extract K closest alternative paths by single-edge removal from optimal.

    For each edge on the optimal path, remove it (set cost to infinity)
    and re-solve. Returns alternatives sorted by cost difference (ascending).

    Parameters
    ----------
    pred_costs : (n_edges,) predicted / posterior-mean edge costs
    grid_size  : n (grid is n x n)
    K          : number of alternatives to return

    Returns
    -------
    opt_path     : (n_edges,) binary — optimal path
    alternatives : list of (alt_path, cost_diff, sym_diff)
        alt_path  : (n_edges,) binary — alternative path
        cost_diff : float — cost of alternative minus optimal
        sym_diff  : (n_edges,) bool — edges in symmetric difference
    """
    costs = np.asarray(pred_costs, dtype=np.float64)
    n = grid_size

    opt_path = solve_shortest_path_edges(costs, n)
    opt_cost = float(costs @ opt_path)
    opt_edges = np.where(opt_path > 0.5)[0]

    alternatives = []
    seen = set()

    for e in opt_edges:
        mod = costs.copy()
        mod[e] = 1e10
        alt = solve_shortest_path_edges(mod, n)
        key = tuple(np.where(alt > 0.5)[0])
        if key in seen:
            continue
        seen.add(key)

        cost_diff = float(costs @ alt) - opt_cost
        sym_diff = np.abs(opt_path - alt) > 0.5
        alternatives.append((alt, cost_diff, sym_diff))

    alternatives.sort(key=lambda x: x[1])
    return opt_path, alternatives[:K]

## Agents

In [49]:
class LinearBayesianAgent:
    """Linear features + per-arm Bayesian regression + UCB/Thompson exploration.

    Supports imputation modes:
      'model'                — posterior mean
      'ucb'                  — posterior mean + beta*sigma
      'thompson'             — averaged posterior samples
      'decision_conditional' — sampling union approach
      'contested'            — Contested EXPLORATION: DP slacks boost UCB/TS
                               toward near-boundary unobserved edges.
                               Gradient is strictly masked (no imputed gradients).
      'disc_explore'         — Discriminative EXPLORATION: K-alternative paths
                               identify discriminating edges; boost UCB on those.
                               Gradient is strictly masked.
      'disc_impute'          — Discriminative IMPUTATION: DX-style exploration
                               + adversarial imputation on discriminating edges
                               (pessimistic on optimal, optimistic on alternative).
                               Gradient on disc edges downweighted by disc_lambda.
      'cx_di'               — CX exploration (slack-based UCB boost) in act()
                               + DI adversarial imputation in update().
                               Best exploration + best gradient signal.

    Adaptive alpha (adaptive_alpha=True):
      Instead of fixed contested_alpha, adapts alpha each round to maintain
      target posterior calibration on observed edges. If coverage > target,
      alpha decreases (posterior is conservative); if < target, alpha increases.
    """

    def __init__(self, context_dim, n_arms, weights, capacity,
                 loss_grad_fn, hidden_dim=32, beta=1.0, lambda_reg=1.0, lr=0.01,
                 weight_decay=1e-4, max_grad_norm=5.0, imputation='ucb',
                 mask_grad=False, minimize=False, solver=None, seed=None,
                 n_thompson_samples=20, thompson_temp=1.0,
                 action_type='ucb',
                 lr_schedule=None,
                 n_relevance_samples=10,
                 dc_impute='thompson',
                 confidence_weight=False,
                 cw_epsilon=1e-3,
                 # --- Parameters for contested/discriminative exploration ---
                 grid_size=None,
                 contested_tau=2.0,
                 contested_eps=1e-6,
                 contested_alpha=2.0,
                 disc_K=5,
                 disc_lambda=1.0,
                 # --- Adaptive alpha ---
                 adaptive_alpha=False,
                 alpha_target=0.68,
                 alpha_lr=0.02,
                 **kwargs):
        self.n_arms = n_arms
        self.weights_np = np.asarray(weights, dtype=np.float64)
        self.capacity = capacity
        self.beta = beta
        self.loss_grad_fn = loss_grad_fn
        self.imputation = imputation
        self.mask_grad = mask_grad
        self.minimize = minimize
        self.n_thompson_samples = n_thompson_samples
        self.thompson_temp = thompson_temp
        self.action_type = action_type
        self.n_relevance_samples = n_relevance_samples
        self.dc_impute = dc_impute
        self.confidence_weight = confidence_weight
        self.cw_epsilon = cw_epsilon
        self.t = 0

        # Contested / discriminative parameters
        self.grid_size = grid_size
        self.contested_tau = contested_tau
        self.contested_eps = contested_eps
        self.contested_alpha = contested_alpha
        self.disc_K = disc_K
        self.disc_lambda = disc_lambda

        # Adaptive alpha
        self.adaptive_alpha = adaptive_alpha
        self.alpha_target = alpha_target
        self.alpha_lr = alpha_lr
        self.alpha_history = []

        if imputation in ('contested', 'disc_explore', 'disc_impute', 'cx_di') and grid_size is None:
            raise ValueError("grid_size is required for imputation=%r" % imputation)

        # Track contested/discriminating set size for diagnostics
        self.contested_history = []

        if solver is None:
            self.solver = lambda v: solve_knapsack(np.asarray(v), self.weights_np, capacity)
        else:
            self.solver = solver

        key = jax.random.PRNGKey(seed if seed is not None else 0)
        key, init_key = jax.random.split(key)
        self.rng_key = key
        self.W1 = jax.random.normal(init_key, (context_dim, hidden_dim)) * 0.1
        self.b1 = jnp.zeros(hidden_dim)

        lr_or_schedule = lr_schedule if lr_schedule is not None else lr
        self.optimizer = optax.chain(
            optax.clip_by_global_norm(max_grad_norm),
            optax.adamw(lr_or_schedule, weight_decay=weight_decay),
        )
        self.opt_state = self.optimizer.init({'W1': self.W1, 'b1': self.b1})
        self._step = _make_step_fn(self.optimizer)

        self.A, self.b_bayes, self.A_inv = init_per_arm_regression(
            n_arms, hidden_dim, lambda_reg)

    def predict_mean(self, X):
        X_j = jnp.asarray(X, dtype=jnp.float32)
        return np.asarray(_fused_predict(X_j, self.W1, self.b1,
                                         self.A_inv, self.b_bayes))

    def act(self, X):
        self.t += 1
        X_j = jnp.asarray(X, dtype=jnp.float32)

        # --- Contested / CX+DI exploration: slack-based UCB boost ---
        if self.imputation in ('contested', 'cx_di'):
            h = X_j @ self.W1 + self.b1
            theta = jnp.einsum('nij,nj->ni', self.A_inv, self.b_bayes)
            means = theta @ h
            Ax = jnp.einsum('nij,j->ni', self.A_inv, h)
            stds = jnp.sqrt(jnp.maximum(jnp.einsum('i,ni->n', h, Ax), 0.0))

            means_np = np.asarray(means, dtype=np.float64)
            stds_np = np.asarray(stds, dtype=np.float64)
            slacks = compute_edge_slacks(means_np, self.grid_size)

            contested = slacks < self.contested_tau * (stds_np + self.contested_eps)
            boost = np.ones(self.n_arms)
            boost_mask = contested & (slacks > 0)
            if boost_mask.any():
                boost[boost_mask] = 1.0 + self.contested_alpha * np.exp(
                    -slacks[boost_mask] / (stds_np[boost_mask] + self.contested_eps))
            boost_j = jnp.array(boost, dtype=jnp.float32)

            if self.action_type == 'thompson':
                self.rng_key, subkey = jax.random.split(self.rng_key)
                z = jax.random.normal(subkey, shape=(self.n_arms,))
                values = means + stds * boost_j * z
            else:  # ucb
                values = means + self.beta * stds * boost_j

            return self.solver(np.array(values))

        # --- Discriminative exploration / imputation: boost on discriminating edges ---
        if self.imputation in ('disc_explore', 'disc_impute'):
            h = X_j @ self.W1 + self.b1
            theta = jnp.einsum('nij,nj->ni', self.A_inv, self.b_bayes)
            means = theta @ h
            Ax = jnp.einsum('nij,j->ni', self.A_inv, h)
            stds = jnp.sqrt(jnp.maximum(jnp.einsum('i,ni->n', h, Ax), 0.0))

            means_np = np.asarray(means, dtype=np.float64)
            stds_np = np.asarray(stds, dtype=np.float64)

            # Extract K alternatives and find discriminating edges
            opt_path, alts = extract_k_alternatives(
                means_np, self.grid_size, self.disc_K)
            disc_edges = np.zeros(self.n_arms, dtype=bool)
            for _, _, sym_diff in alts:
                disc_edges |= sym_diff

            # Boost UCB on off-path discriminating edges
            slacks = compute_edge_slacks(means_np, self.grid_size)
            boost = np.ones(self.n_arms)
            boost_mask = disc_edges & (opt_path < 0.5)
            if boost_mask.any():
                boost[boost_mask] = 1.0 + self.contested_alpha * np.exp(
                    -slacks[boost_mask] / (stds_np[boost_mask] + self.contested_eps))
            boost_j = jnp.array(boost, dtype=jnp.float32)

            values = means + self.beta * stds * boost_j
            return self.solver(np.array(values))

        if self.action_type == 'thompson':
            h = X_j @ self.W1 + self.b1
            theta = jnp.einsum('nij,nj->ni', self.A_inv, self.b_bayes)
            means = theta @ h
            Ax = jnp.einsum('nij,j->ni', self.A_inv, h)
            stds = jnp.sqrt(jnp.maximum(jnp.einsum('i,ni->n', h, Ax), 0.0))
            self.rng_key, subkey = jax.random.split(self.rng_key)
            z = jax.random.normal(subkey, shape=(self.n_arms,))
            values = means + stds * z
            return self.solver(np.array(values))

        values = _fused_ucb(X_j, self.W1, self.b1,
                            self.A_inv, self.b_bayes, self.beta)
        return self.solver(np.array(values))

    def update(self, X, selection, rewards, lr=None, full_feedback=False):
        X_j = jnp.asarray(X, dtype=jnp.float32)
        sel_j = jnp.asarray(selection, dtype=jnp.float32)
        rew_j = jnp.asarray(rewards, dtype=jnp.float32)
        sel_mask = sel_j > 0.5

        if full_feedback:
            obs_mask = jnp.ones(self.n_arms, dtype=jnp.bool_)
        else:
            obs_mask = sel_mask

        h, pred, theta, self.A, self.b_bayes, self.A_inv = \
            _fused_bayesian_core(X_j, self.W1, self.b1,
                self.A, self.b_bayes, self.A_inv,
                rew_j, obs_mask)

        # --- Common stds computation ---
        needs_stds = (self.imputation in ('ucb', 'thompson',
                                          'decision_conditional', 'contested',
                                          'disc_explore', 'disc_impute', 'cx_di')
                      or self.confidence_weight)
        if needs_stds:
            Ax = jnp.einsum('nij,j->ni', self.A_inv, h)
            stds = jnp.sqrt(jnp.maximum(jnp.einsum('i,ni->n', h, Ax), 0.0))
        else:
            stds = None

        # --- Adaptive alpha: calibrate on observed edges ---
        if self.adaptive_alpha and stds is not None:
            obs_np = np.asarray(obs_mask)
            obs_idx = np.where(obs_np)[0]
            if len(obs_idx) > 0:
                errors = np.abs(np.asarray(rew_j)[obs_idx] -
                                np.asarray(pred)[obs_idx])
                thresholds = self.contested_alpha * np.asarray(stds)[obs_idx]
                coverage = float(np.mean(errors < thresholds))
                # Smooth multiplicative update: if coverage > target, shrink alpha
                self.contested_alpha *= float(np.exp(
                    self.alpha_lr * (self.alpha_target - coverage)))
                self.contested_alpha = float(np.clip(
                    self.contested_alpha, 0.1, 5.0))
            self.alpha_history.append(self.contested_alpha)

        # --- Imputation & gradient computation ---
        relevant_mask = None

        # =============================================================
        # Contested exploration: MASKED gradient only.
        # =============================================================
        if self.imputation == 'contested':
            Y_target = jnp.where(obs_mask, rew_j, pred)
            grad_pred = self.loss_grad_fn(pred, Y_target)
            grad_pred = jnp.where(obs_mask, grad_pred, 0.0)

            if stds is not None:
                pred_np = np.asarray(pred, dtype=np.float64)
                stds_np = np.asarray(stds, dtype=np.float64)
                contested, _ = contested_mask_and_slacks(
                    pred_np, stds_np, self.grid_size, self.contested_tau)
                self.contested_history.append(int(contested.sum()))

        # =============================================================
        # Discriminative exploration: MASKED gradient, log disc set size.
        # =============================================================
        elif self.imputation == 'disc_explore':
            Y_target = jnp.where(obs_mask, rew_j, pred)
            grad_pred = self.loss_grad_fn(pred, Y_target)
            grad_pred = jnp.where(obs_mask, grad_pred, 0.0)

            pred_np = np.asarray(pred, dtype=np.float64)
            _, alts = extract_k_alternatives(
                pred_np, self.grid_size, self.disc_K)
            disc_edges = np.zeros(self.n_arms, dtype=bool)
            for _, _, sd in alts:
                disc_edges |= sd
            self.contested_history.append(int(disc_edges.sum()))

        # =============================================================
        # Discriminative imputation (DI / CX+DI): adversarial imputation
        # on disc edges. Per-alternative masking prevents noise dilution.
        # =============================================================
        elif self.imputation in ('disc_impute', 'cx_di'):
            pred_np = np.asarray(pred, dtype=np.float64)
            stds_np = np.asarray(stds, dtype=np.float64)

            opt_path, alts = extract_k_alternatives(
                pred_np, self.grid_size, self.disc_K)

            if not alts:
                # Fallback: masked gradient
                Y_target = jnp.where(obs_mask, rew_j, pred)
                grad_pred = self.loss_grad_fn(pred, Y_target)
                grad_pred = jnp.where(obs_mask, grad_pred, 0.0)
                all_disc = np.zeros(self.n_arms, dtype=bool)
            else:
                grad_pred = jnp.zeros(self.n_arms)
                all_disc = np.zeros(self.n_arms, dtype=bool)

                for alt_path, _, sym_diff in alts:
                    imputed = pred_np.copy()

                    # Pessimistic on optimal-only edges (higher cost for min)
                    opt_only = (opt_path > 0.5) & (alt_path < 0.5)
                    imputed[opt_only] += self.contested_alpha * stds_np[opt_only]

                    # Optimistic on alternative-only edges (lower cost for min)
                    alt_only = (alt_path > 0.5) & (opt_path < 0.5)
                    imputed[alt_only] -= self.contested_alpha * stds_np[alt_only]

                    # Target: observed where available, adversarial elsewhere
                    Y_k = np.where(np.asarray(obs_mask), np.asarray(rew_j),
                                   imputed)
                    grad_k = self.loss_grad_fn(
                        pred, jnp.array(Y_k, dtype=jnp.float32))

                    # Per-alternative masking: observed at full weight,
                    # THIS alternative's disc edges at lambda, rest zero.
                    disc_k = jnp.array(sym_diff)
                    grad_k = jnp.where(
                        obs_mask, grad_k,
                        jnp.where(disc_k, self.disc_lambda * grad_k, 0.0))

                    grad_pred = grad_pred + grad_k
                    all_disc |= sym_diff

                grad_pred = grad_pred / len(alts)

            self.contested_history.append(int(all_disc.sum()))

        # =============================================================
        # Existing imputation modes (unchanged)
        # =============================================================
        elif self.imputation == 'decision_conditional':
            relevant_mask = obs_mask
            for _ in range(self.n_relevance_samples):
                self.rng_key, subkey = jax.random.split(self.rng_key)
                z = jax.random.normal(subkey, shape=(self.n_arms,))
                sampled_costs = pred + stds * z
                sol = self.solver(np.array(sampled_costs))
                relevant_mask = relevant_mask | (jnp.asarray(sol) > 0.5)

            if self.dc_impute == 'thompson':
                grad_pred = jnp.zeros(self.n_arms)
                for _ in range(self.n_thompson_samples):
                    self.rng_key, subkey = jax.random.split(self.rng_key)
                    z = jax.random.normal(subkey, shape=(self.n_arms,))
                    imputed = pred + (stds / self.thompson_temp) * z
                    imputed = jnp.where(relevant_mask, imputed, pred)
                    Y_target = jnp.where(obs_mask, rew_j, imputed)
                    grad_pred = grad_pred + self.loss_grad_fn(pred, Y_target)
                grad_pred = grad_pred / self.n_thompson_samples
            else:
                Y_target = jnp.where(obs_mask, rew_j, pred)
                grad_pred = self.loss_grad_fn(pred, Y_target)

        elif self.imputation == 'thompson':
            grad_pred = jnp.zeros(self.n_arms)
            for _ in range(self.n_thompson_samples):
                self.rng_key, subkey = jax.random.split(self.rng_key)
                z = jax.random.normal(subkey, shape=(self.n_arms,))
                imputed = pred + (stds / self.thompson_temp) * z
                Y_target = jnp.where(obs_mask, rew_j, imputed)
                grad_pred = grad_pred + self.loss_grad_fn(pred, Y_target)
            grad_pred = grad_pred / self.n_thompson_samples
        else:
            if self.imputation == 'ucb':
                imputed = pred + jnp.abs(self.beta) * stds
            else:
                imputed = pred
            Y_target = jnp.where(obs_mask, rew_j, imputed)
            grad_pred = self.loss_grad_fn(pred, Y_target)

        # --- Gradient masking / confidence weighting ---
        # (skip for contested/disc modes — already handled above)
        if self.imputation not in ('contested', 'disc_explore', 'disc_impute', 'cx_di'):
            if self.confidence_weight:
                weights = 1.0 / (stds + self.cw_epsilon)
                weights = weights / jnp.max(weights)
                if relevant_mask is not None:
                    weights = jnp.where(relevant_mask, weights, 0.0)
                grad_pred = grad_pred * weights
            elif self.mask_grad:
                if relevant_mask is not None:
                    grad_pred = jnp.where(relevant_mask, grad_pred, 0.0)
                else:
                    grad_pred = jnp.where(obs_mask, grad_pred, 0.0)

        grads = _bayesian_grads(X_j, theta, grad_pred)

        params = {'W1': self.W1, 'b1': self.b1}
        new_params, self.opt_state = self._step(grads, self.opt_state, params)
        self.W1 = jnp.clip(new_params['W1'], -10, 10)
        self.b1 = new_params['b1']

In [50]:
class RandomAgent:
    """Plays a random feasible solution each round. No learning."""

    def __init__(self, n_arms, weights, capacity, solver=None,
                 minimize=False, seed=None, **kwargs):
        self.n_arms = n_arms
        self.weights_np = np.asarray(weights, dtype=np.float64)
        self.capacity = capacity
        self.minimize = minimize

        if solver is None:
            self.solver = lambda v: solve_knapsack(np.asarray(v), self.weights_np, capacity)
        else:
            self.solver = solver

        self.rng = np.random.default_rng(seed if seed is not None else 0)

    def predict_mean(self, X):
        return np.zeros(self.n_arms)

    def act(self, X):
        random_costs = self.rng.standard_normal(self.n_arms)
        return self.solver(random_costs)

    def update(self, X, selection, rewards, lr=None, full_feedback=False):
        pass  # no learning


class LinearEpsilonGreedyAgent:
    """Linear agent with decaying epsilon-greedy exploration."""

    def __init__(self, context_dim, n_arms, weights, capacity,
                 loss_grad_fn, hidden_dim=32, epsilon0=1.0, lr=0.01,
                 weight_decay=1e-4, max_grad_norm=5.0, mask_grad=False,
                 minimize=False, solver=None, seed=None, lr_schedule=None,
                 **kwargs):
        self.n_arms = n_arms
        self.weights_np = np.asarray(weights, dtype=np.float64)
        self.capacity = capacity
        self.epsilon0 = epsilon0
        self.loss_grad_fn = loss_grad_fn
        self.mask_grad = mask_grad
        self.minimize = minimize
        self.t = 0

        if solver is None:
            self.solver = lambda v: solve_knapsack(np.asarray(v), self.weights_np, capacity)
        else:
            self.solver = solver

        key = jax.random.PRNGKey(seed if seed is not None else 0)
        key, init_key = jax.random.split(key)
        self.rng_key = key

        self.W1 = jax.random.normal(init_key, (context_dim, hidden_dim)) * 0.01
        self.b1 = jnp.zeros(hidden_dim)
        key, init_key = jax.random.split(key)
        self.W2 = jax.random.normal(init_key, (hidden_dim, n_arms)) * 0.01
        self.b2 = jnp.zeros(n_arms)
        self.rng_key = key
        self.rng = np.random.default_rng(seed if seed is not None else 0)

        lr_or_schedule = lr_schedule if lr_schedule is not None else lr
        self.optimizer = optax.chain(
            optax.clip_by_global_norm(max_grad_norm),
            optax.adam(lr_or_schedule),
        )
        params = {'W1': self.W1, 'b1': self.b1, 'W2': self.W2, 'b2': self.b2}
        self.opt_state = self.optimizer.init(params)

    def _predict(self, X):
        h = X @ self.W1 + self.b1
        return h @ self.W2 + self.b2, h

    def predict_mean(self, X):
        X_j = jnp.asarray(X, dtype=jnp.float32)
        pred, _ = self._predict(X_j)
        return np.asarray(pred)

    def act(self, X):
        self.t += 1
        pred = self.predict_mean(X)

        eps = self.epsilon0 / np.sqrt(self.t)
        if self.rng.random() < eps:
            noisy = pred + self.rng.standard_normal(self.n_arms)
            return self.solver(noisy)
        return self.solver(pred)

    def update(self, X, selection, rewards, lr=None, full_feedback=False):
        X_j = jnp.asarray(X, dtype=jnp.float32)
        sel_j = jnp.asarray(selection, dtype=jnp.float32)
        rew_j = jnp.asarray(rewards, dtype=jnp.float32)
        sel_mask = sel_j > 0.5

        if full_feedback:
            obs_mask = jnp.ones(self.n_arms, dtype=jnp.bool_)
        else:
            obs_mask = sel_mask

        pred, h = self._predict(X_j)

        Y_target = jnp.where(obs_mask, rew_j, pred)
        grad_pred = self.loss_grad_fn(pred, Y_target)
        
        if self.mask_grad:
            grad_pred = jnp.where(obs_mask, grad_pred, 0.0)

        grad_W2 = jnp.outer(h, grad_pred)
        grad_b2 = grad_pred
        grad_h = self.W2 @ grad_pred
        grad_W1 = jnp.outer(X_j, grad_h)
        grad_b1 = grad_h

        grads = {'W1': grad_W1, 'b1': grad_b1, 'W2': grad_W2, 'b2': grad_b2}
        params = {'W1': self.W1, 'b1': self.b1, 'W2': self.W2, 'b2': self.b2}

        updates, self.opt_state = self.optimizer.update(grads, self.opt_state, params)
        new_params = optax.apply_updates(params, updates)

        self.W1 = jnp.clip(new_params['W1'], -10, 10)
        self.b1 = new_params['b1']
        self.W2 = jnp.clip(new_params['W2'], -10, 10)
        self.b2 = new_params['b2']


class LinUCB:
    """Bayesian linear UCB with optional random feature projection."""

    def __init__(self, context_dim, n_arms, weights, capacity,
                 beta=1.0, lambda_reg=1.0, solver=None, minimize=False,
                 hidden_dim=None, seed=None, **kwargs):
        self.n_arms = n_arms
        self.weights_np = np.asarray(weights, dtype=np.float64)
        self.capacity = capacity
        self.beta = beta
        self.minimize = minimize

        if solver is None:
            self.solver = lambda v: solve_knapsack(np.asarray(v), self.weights_np, capacity)
        else:
            self.solver = solver

        # Optional random projection: x -> h = x @ W + b (fixed, not trained)
        if hidden_dim is not None:
            key = jax.random.PRNGKey(seed if seed is not None else 0)
            key, init_key = jax.random.split(key)
            self.W_proj = jax.random.normal(init_key, (context_dim, hidden_dim)) * 0.1
            self.b_proj = jnp.zeros(hidden_dim)
            feat_dim = hidden_dim
        else:
            self.W_proj = None
            self.b_proj = None
            feat_dim = context_dim

        self.A, self.b_bayes, self.A_inv = init_per_arm_regression(
            n_arms, feat_dim, lambda_reg)

    def _features(self, X):
        h = jnp.asarray(X, dtype=jnp.float32)
        if self.W_proj is not None:
            h = h @ self.W_proj + self.b_proj
        return h

    def predict_mean(self, X):
        h = self._features(X)
        pred = compute_bayesian_predictions(h, self.A_inv, self.b_bayes)
        return np.asarray(pred)

    def act(self, X):
        h = self._features(X)
        ucb = compute_ucb_values(h, self.A_inv, self.b_bayes, self.beta)
        return self.solver(np.asarray(ucb))

    def update(self, X, selection, rewards, lr=None, full_feedback=False):
        h = self._features(X)
        rew_j = jnp.asarray(rewards, dtype=jnp.float32)
        mask = jnp.asarray(selection, dtype=jnp.float32) > 0.5
        if full_feedback:
            mask = jnp.ones(self.n_arms, dtype=jnp.bool_)
        self.A, self.b_bayes, self.A_inv = update_bayesian_stats(
            self.A, self.b_bayes, self.A_inv, h, rew_j, mask)

## Experiment Setup

In [51]:
# --- Dataset configurations ---
NOISE_LEVELS = [0.25]

DATASETS = {}
for nw in NOISE_LEVELS:
    nw_tag = str(nw).replace('.', '')
    DATASETS[f'sp_nw{nw_tag}'] = dict(
        env_cls=ShortestPathEdgeEnv,
        env_kwargs=dict(grid_size=20, context_dim=5, noise_width=nw,
                        nonstationarity='none'),
        minimize=True,
        group='shortest_path',
        noise=nw,
    )

HP = dict(lr=0.001, hidden_dim=32, lambda_reg=1.0, beta=1.0, epsilon0=1.0)

N_ROUNDS = 2000
SEEDS = [0, 1, 2, 5, 6]
EVAL_INTERVAL = 100

## Run Experiments

In [52]:
def make_agents(env, seed, minimize):
    solver = env.solve
    bayes_beta = -HP['beta'] if minimize else HP['beta']

    spo = make_spo_grad(solver, is_maximize=not minimize)

    common = dict(context_dim=env.context_dim, n_arms=env.n_arms,
                  weights=env.weights, capacity=env.capacity)
    bayes = dict(hidden_dim=HP['hidden_dim'], lr=HP['lr'], seed=seed,
                 beta=bayes_beta, lambda_reg=HP['lambda_reg'], solver=solver,
                 minimize=minimize)

    sched = make_inv_sqrt_schedule(HP['lr'])

    grid_size = getattr(env, 'grid_size', None)

    agents = {
        # === Baselines ===
        'BayesianMSE': LinearBayesianAgent(**common, **bayes, loss_grad_fn=mse_grad,
                                           imputation='model', lr_schedule=sched),
        'UCB_SPO_ModelMask': LinearBayesianAgent(**common, **bayes, loss_grad_fn=spo,
                                                 action_type='ucb', imputation='model',
                                                 mask_grad=True, lr_schedule=sched),
        'UCB_SPO_TSMask': LinearBayesianAgent(**common, **bayes, loss_grad_fn=spo,
                                              action_type='ucb', imputation='thompson',
                                              mask_grad=True, lr_schedule=sched),
        'TS_SPO_TSMask': LinearBayesianAgent(**common, **bayes, loss_grad_fn=spo,
                                             action_type='thompson', imputation='thompson',
                                             mask_grad=True, lr_schedule=sched),
    }

    if grid_size is not None:
        # === CX reference (best from previous experiments) ===
        agents['UCB_CX_a10'] = LinearBayesianAgent(
            **common, **bayes, loss_grad_fn=spo,
            action_type='ucb', imputation='contested',
            grid_size=grid_size, contested_tau=2.0, contested_alpha=1.0,
            lr_schedule=sched,
        )

        # === Discriminative Exploration (safe: masked gradient) ===
        for alpha in [0.5, 1.0]:
            tag = str(alpha).replace('.', '')
            agents[f'UCB_DX_a{tag}'] = LinearBayesianAgent(
                **common, **bayes, loss_grad_fn=spo,
                action_type='ucb', imputation='disc_explore',
                grid_size=grid_size, contested_alpha=alpha,
                disc_K=5, lr_schedule=sched,
            )

        # === Discriminative Imputation v2 ===
        for lam in [0.1, 0.3, 0.5]:
            tag = str(lam).replace('.', '')
            agents[f'UCB_DI_l{tag}'] = LinearBayesianAgent(
                **common, **bayes, loss_grad_fn=spo,
                action_type='ucb', imputation='disc_impute',
                grid_size=grid_size, contested_alpha=0.5,
                disc_K=5, disc_lambda=lam, lr_schedule=sched,
            )

        # === CX+DI hybrid: CX exploration + DI gradient (fixed alpha=1.0) ===
        for lam in [0.1, 0.3, 0.5]:
            tag = str(lam).replace('.', '')
            agents[f'CX_DI_l{tag}'] = LinearBayesianAgent(
                **common, **bayes, loss_grad_fn=spo,
                action_type='ucb', imputation='cx_di',
                grid_size=grid_size, contested_tau=2.0,
                contested_alpha=1.0, disc_K=5, disc_lambda=lam,
                lr_schedule=sched,
            )

        # === CX+DI with adaptive alpha (init=1.0, target 68% coverage) ===
        for lam in [0.1, 0.3, 0.5]:
            tag = str(lam).replace('.', '')
            agents[f'CXDIa_l{tag}'] = LinearBayesianAgent(
                **common, **bayes, loss_grad_fn=spo,
                action_type='ucb', imputation='cx_di',
                grid_size=grid_size, contested_tau=2.0,
                contested_alpha=1.0, disc_K=5, disc_lambda=lam,
                adaptive_alpha=True, alpha_target=0.68, alpha_lr=0.02,
                lr_schedule=sched,
            )

    return agents


all_agents = {}
all_results = {}

for ds_name, ds_cfg in DATASETS.items():
    env_cls = ds_cfg['env_cls']
    env_kwargs = ds_cfg['env_kwargs']
    minimize = ds_cfg['minimize']

    print(f'\n=== {ds_name} (minimize={minimize}) ===')
    all_results[ds_name] = {}
    all_agents[ds_name] = {}

    for seed in SEEDS:
        print(f"Seed: {seed}")
        agents = make_agents(env_cls(**env_kwargs, seed=seed), seed, minimize=minimize)

        for agent_name, agent in agents.items():
            if agent_name not in all_results[ds_name]:
                all_results[ds_name][agent_name] = {}
                all_agents[ds_name][agent_name] = {}

            env_run = env_cls(**env_kwargs, seed=seed)
            result = run_experiment(
                env_run, agent, n_rounds=N_ROUNDS,
                lr=HP['lr'], full_feedback=False,
                eval_interval=EVAL_INTERVAL)
            all_results[ds_name][agent_name][seed] = result
            all_agents[ds_name][agent_name][seed] = agent
            print(f'  {agent_name:<20s} seed={seed}  final_regret={result["final_regret"]:.1f}')


=== sp_nw025 (minimize=True) ===
Seed: 0
  BayesianMSE          seed=0  final_regret=37065.2
  UCB_SPO_ModelMask    seed=0  final_regret=34673.8
  UCB_SPO_TSMask       seed=0  final_regret=34548.6
  TS_SPO_TSMask        seed=0  final_regret=35324.7
  UCB_CX_a10           seed=0  final_regret=32876.6
  UCB_DX_a05           seed=0  final_regret=33087.4
  UCB_DX_a10           seed=0  final_regret=34168.2
  UCB_DI_l01           seed=0  final_regret=33086.6
  UCB_DI_l03           seed=0  final_regret=33299.3
  UCB_DI_l05           seed=0  final_regret=33418.9
  CX_DI_l01            seed=0  final_regret=33020.9
  CX_DI_l03            seed=0  final_regret=32973.4
  CX_DI_l05            seed=0  final_regret=33150.1
  CXDIa_l01            seed=0  final_regret=33657.4
  CXDIa_l03            seed=0  final_regret=33757.8
  CXDIa_l05            seed=0  final_regret=33928.9
Seed: 1
  BayesianMSE          seed=1  final_regret=32589.5
  UCB_SPO_ModelMask    seed=1  final_regret=30019.4
  UCB_SPO_TSMa

## Plot Results

In [53]:
from lets_plot import gggrid, ggsave
import os

FIGDIR = 'lets-plot-images'
os.makedirs(FIGDIR, exist_ok=True)

# --- Display name mapping (internal -> paper name) ---
DISPLAY_NAMES = {
    'SPO_sched': 'BayesianSPO',
    'MSE_sched': 'BayesianMSE',
    'EG_SPO':    'SPO',
    'EG_MSE':    'MSE',
    'LinUCB':    'LinUCB',
    'Random':    'Random',
    'Greedy':    'Greedy',
}

AGENT_ORDER = [
    'BayesianMSE',
    # Baselines
    'UCB_SPO_TSMask', 'UCB_SPO_ModelMask',
    'TS_SPO_TSMask',
    # CX reference
    'UCB_CX_a10',
    # Discriminative exploration (safe)
    'UCB_DX_a05', 'UCB_DX_a10',
    # Discriminative imputation (DX exploration + DI gradient)
    'UCB_DI_l01', 'UCB_DI_l03', 'UCB_DI_l05',
    # CX+DI hybrid (CX exploration + DI gradient, fixed alpha)
    'CX_DI_l01', 'CX_DI_l03', 'CX_DI_l05',
    # CX+DI adaptive alpha
    'CXDIa_l01', 'CXDIa_l03', 'CXDIa_l05',
]

COLORS = {
    'BayesianMSE':       '#d62728',  # red
    # Baselines
    'UCB_SPO_TSMask':    '#1f77b4',  # blue
    'UCB_SPO_ModelMask': '#2ca02c',  # green
    'TS_SPO_TSMask':     '#9467bd',  # purple
    # CX reference
    'UCB_CX_a10':        '#8c564b',  # brown
    # Discriminative exploration (warm tones)
    'UCB_DX_a05':        '#e6550d',  # dark orange
    'UCB_DX_a10':        '#fd8d3c',  # orange
    # Discriminative imputation (cool tones)
    'UCB_DI_l01':        '#08519c',  # dark steel blue
    'UCB_DI_l03':        '#3182bd',  # steel blue
    'UCB_DI_l05':        '#6baed6',  # light steel blue
    # CX+DI hybrid (green tones)
    'CX_DI_l01':         '#006d2c',  # dark green
    'CX_DI_l03':         '#31a354',  # green
    'CX_DI_l05':         '#74c476',  # light green
    # CX+DI adaptive (magenta/pink tones)
    'CXDIa_l01':         '#7a0177',  # dark magenta
    'CXDIa_l03':         '#c51b8a',  # magenta
    'CXDIa_l05':         '#f768a1',  # pink
}

LINETYPES = {
    'BayesianMSE':       'solid',
    # Baselines
    'UCB_SPO_TSMask':    'solid',
    'UCB_SPO_ModelMask': 'solid',
    'TS_SPO_TSMask':     'dashed',
    # CX reference
    'UCB_CX_a10':        'dotdash',
    # Discriminative exploration
    'UCB_DX_a05':        'dashed',
    'UCB_DX_a10':        'dashed',
    # Discriminative imputation
    'UCB_DI_l01':        'dotted',
    'UCB_DI_l03':        'dotted',
    'UCB_DI_l05':        'dotted',
    # CX+DI hybrid
    'CX_DI_l01':         'twodash',
    'CX_DI_l03':         'twodash',
    'CX_DI_l05':         'twodash',
    # CX+DI adaptive
    'CXDIa_l01':         'longdash',
    'CXDIa_l03':         'longdash',
    'CXDIa_l05':         'longdash',
}

EXCLUDE_FROM_LINES = []

WINDOW = 50

LINE_LEGEND_THEME = theme(legend_position='bottom', legend_title=element_blank(),
                          legend_text=element_text(size=9))

# --- Build per-seed data (with display names) ---
rows_cum, rows_pr, rows_box = [], [], []
for ds_name, ds_cfg in DATASETS.items():
    ds_results = all_results[ds_name]
    for agent_name, seed_results in ds_results.items():
        dname = DISPLAY_NAMES.get(agent_name, agent_name)
        for seed in SEEDS:
            result = seed_results[seed]
            for t, r in enumerate(result['cumulative_regret'], 1):
                rows_cum.append(dict(dataset=ds_name, agent=dname,
                                     seed=seed, round=t, cumulative_regret=r))
            kernel = np.ones(WINDOW) / WINDOW
            smoothed = np.convolve(result['regrets'], kernel, mode='valid')
            for t, r in enumerate(smoothed):
                rows_pr.append(dict(dataset=ds_name, agent=dname,
                                    seed=seed, round=t + WINDOW, per_round_regret=r))
            rows_box.append(dict(
                dataset=ds_name, agent=dname, seed=seed,
                group=ds_cfg['group'], noise=ds_cfg['noise'],
                final_cumulative_regret=result['final_regret'],
                last_round_regret=np.mean(result['regrets'][-WINDOW:]),
            ))

df_cum = pd.DataFrame(rows_cum)
df_cum_agg = df_cum.groupby(['dataset', 'agent', 'round']).agg(
    mean=('cumulative_regret', 'mean'), std=('cumulative_regret', 'std')
).reset_index()
df_cum_agg['ymin'] = df_cum_agg['mean'] - df_cum_agg['std']
df_cum_agg['ymax'] = df_cum_agg['mean'] + df_cum_agg['std']

df_pr = pd.DataFrame(rows_pr)
df_pr_agg = df_pr.groupby(['dataset', 'agent', 'round']).agg(
    mean=('per_round_regret', 'mean'), std=('per_round_regret', 'std')
).reset_index()
df_pr_agg['ymin'] = df_pr_agg['mean'] - df_pr_agg['std']
df_pr_agg['ymax'] = df_pr_agg['mean'] + df_pr_agg['std']

df_box = pd.DataFrame(rows_box)

# Ordered list of all agents present in results (display names)
all_agent_set = set()
for ds_name in DATASETS:
    all_agent_set.update(DISPLAY_NAMES.get(a, a) for a in all_results[ds_name].keys())
legend_agents = [a for a in AGENT_ORDER if a in all_agent_set]

# Line plot agents only — used for legend filtering
LINE_COLORS = {k: v for k, v in COLORS.items() if k not in EXCLUDE_FROM_LINES}
LINE_LINETYPES = {k: v for k, v in LINETYPES.items() if k not in EXCLUDE_FROM_LINES}

# --- Layout 1: flat gggrid, legend on line plots only, no legend on boxplots ---
W = 1800
ROW_H = 500

all_plots = []
for ds_name in DATASETS:
    d_cum = df_cum_agg[df_cum_agg['dataset'] == ds_name]
    d_pr = df_pr_agg[df_pr_agg['dataset'] == ds_name]
    d_box = df_box[df_box['dataset'] == ds_name].copy()
    noise = DATASETS[ds_name]['noise']
    group = DATASETS[ds_name]['group']
    label = f'{group} (noise={noise})'

    order = (d_box.groupby('agent')['final_cumulative_regret']
             .median().sort_values().index.tolist())
    d_box['agent'] = pd.Categorical(d_box['agent'], categories=order, ordered=True)

    d_cum_filt = d_cum[~d_cum['agent'].isin(EXCLUDE_FROM_LINES)]
    d_pr_filt = d_pr[~d_pr['agent'].isin(EXCLUDE_FROM_LINES)]

    p_cum = (ggplot(d_cum_filt, aes(x='round', y='mean', color='agent', fill='agent'))
        + geom_ribbon(aes(ymin='ymin', ymax='ymax'), alpha=0.15, color='rgba(0,0,0,0)')
        + geom_line(aes(linetype='agent'), size=1.0)
        + scale_color_manual(values=LINE_COLORS) + scale_fill_manual(values=LINE_COLORS)
        + scale_linetype_manual(values=LINE_LINETYPES)
        + guides(color=guide_legend(nrow=4), fill='none', linetype=guide_legend(nrow=4))
        + labs(title=f'{label}: Cumulative Regret', x='Round', y='Cumulative Regret')
        + theme_minimal() + LINE_LEGEND_THEME)

    p_pr = (ggplot(d_pr_filt, aes(x='round', y='mean', color='agent', fill='agent'))
        + geom_ribbon(aes(ymin='ymin', ymax='ymax'), alpha=0.15, color='rgba(0,0,0,0)')
        + geom_line(aes(linetype='agent'), size=1.0)
        + scale_color_manual(values=LINE_COLORS) + scale_fill_manual(values=LINE_COLORS)
        + scale_linetype_manual(values=LINE_LINETYPES)
        + guides(color=guide_legend(nrow=4), fill='none', linetype=guide_legend(nrow=4))
        + labs(title=f'{label}: Per-Round Regret ({WINDOW}-avg)', x='Round', y='Per-Round Regret')
        + theme_minimal() + LINE_LEGEND_THEME)

    p_box_cum = (ggplot(d_box, aes(x='agent', y='final_cumulative_regret', fill='agent'))
        + geom_boxplot(alpha=0.7, width=0.6)
        + geom_jitter(aes(color='agent'), width=0.15, size=2.5, alpha=0.8)
        + scale_fill_manual(values=COLORS) + scale_color_manual(values=COLORS)
        + labs(title=f'{label}: Final Cum. Regret', x='', y='Cumulative Regret')
        + theme_minimal()
        + theme(legend_position='none', axis_text_x=element_text(angle=45, hjust=1)))

    p_box_pr = (ggplot(d_box, aes(x='agent', y='last_round_regret', fill='agent'))
        + geom_boxplot(alpha=0.7, width=0.6)
        + geom_jitter(aes(color='agent'), width=0.15, size=2.5, alpha=0.8)
        + scale_fill_manual(values=COLORS) + scale_color_manual(values=COLORS)
        + labs(title=f'{label}: Avg Last-{WINDOW} Regret', x='', y='Per-Round Regret')
        + theme_minimal()
        + theme(legend_position='none', axis_text_x=element_text(angle=45, hjust=1)))

    all_plots.extend([p_cum, p_pr, p_box_cum, p_box_pr])

n_datasets = len(DATASETS)
total_h = ROW_H * n_datasets

fig_layout1 = gggrid(all_plots, ncol=4, align=True, hspace=30, vspace=40) + ggsize(W, total_h)
fig_layout1.show()

## Summary Statistics

In [54]:
for ds_name, ds_results in all_results.items():
    print(f'\n{ds_name}')
    print(f'{"Agent":<22s} {"Final Regret":>14s} {"% vs MSE":>10s}')
    print('-' * 50)
    
    mse_final = np.mean([ds_results['BayesianMSE'][s]['final_regret'] for s in SEEDS])
    
    for agent_name in AGENT_ORDER:
        if agent_name not in ds_results:
            continue
        seed_results = ds_results[agent_name]
        final = np.array([seed_results[s]['final_regret'] for s in SEEDS])
        pct_vs_mse = (final.mean() - mse_final) / mse_final * 100
        sign = '+' if pct_vs_mse >= 0 else ''
        print(f'  {agent_name:<20s} {final.mean():>8.1f} +/- {final.std():>5.1f}'
              f'  {sign}{pct_vs_mse:.1f}%')

# --- Cross-dataset summary ---
print('\n\n=== Cross-Dataset Summary: % vs BayesianMSE ===')
agents_show = [a for a in AGENT_ORDER if a != 'BayesianMSE']
header = f'{"Agent":<22s}'
for ds_name in DATASETS:
    header += f' {ds_name:>15s}'
print(header)
print('-' * (22 + 16 * len(DATASETS)))

for agent_name in agents_show:
    row = f'{agent_name:<22s}'
    for ds_name in DATASETS:
        if agent_name not in all_results.get(ds_name, {}):
            row += f' {"N/A":>15s}'
            continue
        mse_final = np.mean([all_results[ds_name]['BayesianMSE'][s]['final_regret'] for s in SEEDS])
        final = np.mean([all_results[ds_name][agent_name][s]['final_regret'] for s in SEEDS])
        pct = (final - mse_final) / mse_final * 100
        sign = '+' if pct >= 0 else ''
        row += f' {sign}{pct:>13.1f}%'
    print(row)


sp_nw025
Agent                    Final Regret   % vs MSE
--------------------------------------------------
  BayesianMSE           35383.7 +/- 2195.6  +0.0%
  UCB_SPO_TSMask        32243.7 +/- 2059.3  -8.9%
  UCB_SPO_ModelMask     32536.6 +/- 1865.7  -8.0%
  TS_SPO_TSMask         33384.2 +/- 1376.7  -5.7%
  UCB_CX_a10            31882.1 +/- 1560.2  -9.9%
  UCB_DX_a05            32327.6 +/- 1563.5  -8.6%
  UCB_DX_a10            32549.2 +/- 1814.1  -8.0%
  UCB_DI_l01            32174.6 +/- 1818.4  -9.1%
  UCB_DI_l03            32315.7 +/- 1781.3  -8.7%
  UCB_DI_l05            32540.3 +/- 1781.1  -8.0%
  CX_DI_l01             32035.5 +/- 1632.8  -9.5%
  CX_DI_l03             32210.8 +/- 1751.4  -9.0%
  CX_DI_l05             32382.4 +/- 1688.7  -8.5%
  CXDIa_l01             32924.2 +/- 1906.9  -7.0%
  CXDIa_l03             32939.3 +/- 1848.9  -6.9%
  CXDIa_l05             33204.0 +/- 1841.4  -6.2%


=== Cross-Dataset Summary: % vs BayesianMSE ===
Agent                         sp_nw025
-

## Alternative Visualization: Noise Comparison

Each figure shows one metric type. Rows = dataset groups (shortest_path, knapsack). Columns = noise levels (0, 0.5, 1.0). Shared y-axes within each row to show relative noise impact.

In [55]:
# --- Layout 2: Group by dataset type, columns = noise levels ---
# Separate figure per metric type, shared y-axes within each group row.

from collections import defaultdict
groups = defaultdict(list)  # group_name -> [(noise, ds_name), ...]
for ds_name, ds_cfg in DATASETS.items():
    groups[ds_cfg['group']].append((ds_cfg['noise'], ds_name))
for g in groups:
    groups[g].sort()

group_names = sorted(groups.keys())
n_noise = max(len(v) for v in groups.values())

w2 = 500 * n_noise
h2_row = 500

# ====== Figure 1: Cumulative Regret Lines ======
plots_cum = []
for grp in group_names:
    ymax_grp = 0
    for noise, ds_name in groups[grp]:
        d = df_cum_agg[(df_cum_agg['dataset'] == ds_name) &
                       (~df_cum_agg['agent'].isin(EXCLUDE_FROM_LINES))]
        ymax_grp = max(ymax_grp, float(d['ymax'].max()))
    ylim_grp = [0, ymax_grp * 1.05]

    for noise, ds_name in groups[grp]:
        d = df_cum_agg[(df_cum_agg['dataset'] == ds_name) &
                       (~df_cum_agg['agent'].isin(EXCLUDE_FROM_LINES))]
        p = (ggplot(d, aes(x='round', y='mean', color='agent', fill='agent'))
            + geom_ribbon(aes(ymin='ymin', ymax='ymax'), alpha=0.15, color='rgba(0,0,0,0)')
            + geom_line(aes(linetype='agent'), size=1.0)
            + scale_color_manual(values=LINE_COLORS) + scale_fill_manual(values=LINE_COLORS)
            + scale_linetype_manual(values=LINE_LINETYPES)
            + scale_y_continuous(limits=ylim_grp)
            + guides(color=guide_legend(nrow=2), fill='none', linetype=guide_legend(nrow=2))
            + labs(title=f'{grp} (noise={noise})', x='Round', y='Cumulative Regret')
            + theme_minimal() + LINE_LEGEND_THEME)
        plots_cum.append(p)

print('=== Cumulative Regret (noise comparison) ===')
fig_cum = gggrid(plots_cum, ncol=n_noise, align=True, hspace=30, vspace=40) + ggsize(w2, h2_row * len(group_names))
fig_cum.show()
ggsave(fig_cum, os.path.join(FIGDIR, 'noise_cumulative_regret.svg'))

# ====== Figure 2: Per-Round Regret Lines ======
plots_pr = []
for grp in group_names:
    ymax_grp = 0
    for noise, ds_name in groups[grp]:
        d = df_pr_agg[(df_pr_agg['dataset'] == ds_name) &
                      (~df_pr_agg['agent'].isin(EXCLUDE_FROM_LINES))]
        ymax_grp = max(ymax_grp, float(d['ymax'].max()))
    ylim_grp = [0, ymax_grp * 1.05]

    for noise, ds_name in groups[grp]:
        d = df_pr_agg[(df_pr_agg['dataset'] == ds_name) &
                      (~df_pr_agg['agent'].isin(EXCLUDE_FROM_LINES))]
        p = (ggplot(d, aes(x='round', y='mean', color='agent', fill='agent'))
            + geom_ribbon(aes(ymin='ymin', ymax='ymax'), alpha=0.15, color='rgba(0,0,0,0)')
            + geom_line(aes(linetype='agent'), size=1.0)
            + scale_color_manual(values=LINE_COLORS) + scale_fill_manual(values=LINE_COLORS)
            + scale_linetype_manual(values=LINE_LINETYPES)
            + scale_y_continuous(limits=ylim_grp)
            + guides(color=guide_legend(nrow=2), fill='none', linetype=guide_legend(nrow=2))
            + labs(title=f'{grp} (noise={noise})', x='Round',
                   y=f'Per-Round Regret ({WINDOW}-avg)')
            + theme_minimal() + LINE_LEGEND_THEME)
        plots_pr.append(p)

print('=== Per-Round Regret (noise comparison) ===')
fig_pr = gggrid(plots_pr, ncol=n_noise, align=True, hspace=30, vspace=40) + ggsize(w2, h2_row * len(group_names))
fig_pr.show()
ggsave(fig_pr, os.path.join(FIGDIR, 'noise_per_round_regret.svg'))

# ====== Figure 3: Final Cumulative Regret Boxplot ======
plots_box_cum = []
for grp in group_names:
    d_grp = df_box[df_box['group'] == grp]
    ymax_grp = float(d_grp['final_cumulative_regret'].max()) * 1.1
    ylim_grp = [0, ymax_grp]
    order = (d_grp.groupby('agent')['final_cumulative_regret']
             .median().sort_values().index.tolist())

    for noise, ds_name in groups[grp]:
        d = df_box[df_box['dataset'] == ds_name].copy()
        d['agent'] = pd.Categorical(d['agent'], categories=order, ordered=True)
        p = (ggplot(d, aes(x='agent', y='final_cumulative_regret', fill='agent'))
            + geom_boxplot(alpha=0.7, width=0.6)
            + geom_jitter(aes(color='agent'), width=0.15, size=2.5, alpha=0.8)
            + scale_fill_manual(values=COLORS) + scale_color_manual(values=COLORS)
            + scale_y_continuous(limits=ylim_grp)
            + labs(title=f'{grp} (noise={noise})', x='', y='Cumulative Regret')
            + theme_minimal()
            + theme(legend_position='none',
                    axis_text_x=element_text(angle=45, hjust=1)))
        plots_box_cum.append(p)

print('=== Final Cumulative Regret Boxplot (noise comparison) ===')
fig_box_cum = gggrid(plots_box_cum, ncol=n_noise, align=True, hspace=30, vspace=40) + ggsize(w2, h2_row * len(group_names))
fig_box_cum.show()
ggsave(fig_box_cum, os.path.join(FIGDIR, 'noise_boxplot_cumulative.svg'))

# ====== Figure 4: Avg Last-WINDOW Regret Boxplot ======
plots_box_pr = []
for grp in group_names:
    d_grp = df_box[df_box['group'] == grp]
    ymax_grp = float(d_grp['last_round_regret'].max()) * 1.1
    ylim_grp = [0, ymax_grp]
    order = (d_grp.groupby('agent')['last_round_regret']
             .median().sort_values().index.tolist())

    for noise, ds_name in groups[grp]:
        d = df_box[df_box['dataset'] == ds_name].copy()
        d['agent'] = pd.Categorical(d['agent'], categories=order, ordered=True)
        p = (ggplot(d, aes(x='agent', y='last_round_regret', fill='agent'))
            + geom_boxplot(alpha=0.7, width=0.6)
            + geom_jitter(aes(color='agent'), width=0.15, size=2.5, alpha=0.8)
            + scale_fill_manual(values=COLORS) + scale_color_manual(values=COLORS)
            + scale_y_continuous(limits=ylim_grp)
            + labs(title=f'{grp} (noise={noise})', x='', y='Per-Round Regret')
            + theme_minimal()
            + theme(legend_position='none',
                    axis_text_x=element_text(angle=45, hjust=1)))
        plots_box_pr.append(p)

print(f'=== Avg Last-{WINDOW} Regret Boxplot (noise comparison) ===')
fig_box_pr = gggrid(plots_box_pr, ncol=n_noise, align=True, hspace=30, vspace=40) + ggsize(w2, h2_row * len(group_names))
fig_box_pr.show()
ggsave(fig_box_pr, os.path.join(FIGDIR, 'noise_boxplot_last_round.svg'))

print(f'\nAll figures saved to {FIGDIR}/')

=== Cumulative Regret (noise comparison) ===


=== Per-Round Regret (noise comparison) ===


=== Final Cumulative Regret Boxplot (noise comparison) ===


=== Avg Last-50 Regret Boxplot (noise comparison) ===



All figures saved to lets-plot-images/


In [56]:
# =========================================================================
# Contested / discriminating set diagnostic: set size over time
# =========================================================================

from collections import defaultdict

fig_contested = []
for ds_name in DATASETS:
    rows_c = []
    for agent_name, seed_agents in all_agents[ds_name].items():
        if not any(tag in agent_name for tag in ('CX', 'DX', 'DI')):
            continue
        for seed, agent in seed_agents.items():
            if hasattr(agent, 'contested_history') and agent.contested_history:
                for t, nc in enumerate(agent.contested_history, 1):
                    rows_c.append(dict(agent=agent_name, seed=seed,
                                       round=t, n_contested=nc))

    if rows_c:
        df_c = pd.DataFrame(rows_c)
        df_c_agg = df_c.groupby(['agent', 'round']).agg(
            mean=('n_contested', 'mean'),
            std=('n_contested', 'std'),
        ).reset_index()
        df_c_agg['ymin'] = df_c_agg['mean'] - df_c_agg['std']
        df_c_agg['ymax'] = df_c_agg['mean'] + df_c_agg['std']

        p = (ggplot(df_c_agg, aes(x='round', y='mean', color='agent'))
            + geom_ribbon(aes(ymin='ymin', ymax='ymax', fill='agent'),
                          alpha=0.15, color='rgba(0,0,0,0)')
            + geom_line(size=1.0)
            + labs(title=f'{ds_name}: Contested/Discriminating Set Size Over Time',
                   x='Round', y='|Set|')
            + theme_minimal()
            + theme(legend_position='bottom'))
        p.show()

# =========================================================================
# Adaptive alpha diagnostic: alpha trajectory over time
# =========================================================================

for ds_name in DATASETS:
    rows_a = []
    for agent_name, seed_agents in all_agents[ds_name].items():
        for seed, agent in seed_agents.items():
            if hasattr(agent, 'alpha_history') and agent.alpha_history:
                for t, alpha in enumerate(agent.alpha_history, 1):
                    rows_a.append(dict(agent=agent_name, seed=seed,
                                       round=t, alpha=alpha))

    if rows_a:
        df_a = pd.DataFrame(rows_a)
        df_a_agg = df_a.groupby(['agent', 'round']).agg(
            mean=('alpha', 'mean'),
            std=('alpha', 'std'),
        ).reset_index()
        df_a_agg['ymin'] = df_a_agg['mean'] - df_a_agg['std']
        df_a_agg['ymax'] = df_a_agg['mean'] + df_a_agg['std']

        p = (ggplot(df_a_agg, aes(x='round', y='mean', color='agent'))
            + geom_ribbon(aes(ymin='ymin', ymax='ymax', fill='agent'),
                          alpha=0.15, color='rgba(0,0,0,0)')
            + geom_line(size=1.0)
            + geom_hline(yintercept=1.0, linetype='dashed', color='gray', alpha=0.5)
            + labs(title=f'{ds_name}: Adaptive Alpha Trajectory',
                   x='Round', y='alpha')
            + theme_minimal()
            + theme(legend_position='bottom'))
        p.show()